# 01 — Exploratory analysis of simulated alerts

Inspect PLAsTiCC light curves, class imbalance, measurement quality and the
differences between the labelled training sample and the test population.
The test labels are used only for descriptive evaluation, not model fitting.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
FIGDIR = config.FIGURE_DIR


CLASS_NAMES = {6:'μLens', 15:'TDE', 16:'EB', 42:'SNII', 52:'SNIax', 53:'Mira',
               62:'SNIbc', 64:'KN', 65:'M-dwarf', 67:'SNIa-91bg', 88:'AGN',
               90:'SNIa', 92:'RRL', 95:'SLSN-I', 991:'novel-1', 992:'novel-2',
               993:'novel-3', 994:'novel-4'}
BANDS = list('ugrizy')
BAND_COLORS = {0:'#6a3d9a', 1:'#1b9e77', 2:'#d62728', 3:'#8c2d04', 4:'#7f5539', 5:'#525252'}
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':0.25})


## 1. Load the original data


In [ ]:
df = pd.read_csv(config.TRAIN_CSV)
metadata_table = pd.read_csv(config.METADATA_CSV)
print(f'{len(df):,} detecciones · {df.object_id.nunique():,} objects · {metadata_table.target.nunique()} classes')
df.head()


## 2. Class imbalance


In [ ]:

vc = metadata_table['target'].value_counts()
labels = [f"{CLASS_NAMES.get(t, t)}" for t in vc.index]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, vc.values, color='#4C72B0')
ax.set_yscale('log')
ax.set_ylabel('Number of objects (log)'); ax.set_xlabel('Class')
ax.set_title('Class distribution in the training set')
for x, v in zip(labels, vc.values):
    ax.text(x, v*1.05, f'{v:,}', ha='center', va='bottom', fontsize=7)
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
fig.savefig(FIGDIR / '01_desbalance_clases.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Example multiband light curves


In [ ]:

ejemplos = {}
for cls in [90, 16, 92]:
    object_ids = metadata_table[metadata_table.target == cls].object_id
    sub = df[df.object_id.isin(object_ids)]
    oid = sub.groupby('object_id').size().idxmax()  
    ejemplos[cls] = oid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (cls, oid) in zip(axes, ejemplos.items()):
    c = df[df.object_id == oid]
    for b in range(6):
        s = c[c.passband == b]
        ax.errorbar(s.mjd, s.flux, yerr=s.flux_err, fmt='o', ms=3, lw=.6,
                    color=BAND_COLORS[b], alpha=.7, label=BANDS[b])
    ax.axhline(0, color='k', lw=.6, ls='--')
    ax.set_title(f"{CLASS_NAMES[cls]} · object {oid}")
    ax.set_xlabel('MJD'); ax.set_ylabel('Flux')
axes[0].legend(ncol=3, fontsize=7, title='Band')
plt.tight_layout()
fig.savefig(FIGDIR / '01_curvas_ejemplo.png', dpi=150, bbox_inches='tight')
fig.savefig(FIGDIR / '01_curvas_ejemplo.pdf', bbox_inches='tight')
plt.show()


## 4. Measurement quality

Inspect flux, signal-to-noise ratio and negative differential-flux measurements.
Negative flux can contain information; it is not removed merely for its sign.


In [ ]:

frac_neg = (df.flux < 0).mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(df.flux, -500, 1500), bins=200, color='#55A868')
ax.axvline(0, color='k', lw=1)
ax.set_yscale('log')
ax.set_xlabel('Flux (clipped to [-500, 1500] for display)')
ax.set_ylabel('Number of observations (log)')
ax.set_title(f'Flux distribution — {frac_neg:.0%} of measurements are negative')
plt.tight_layout()
fig.savefig(FIGDIR / '01_flujo_negativo.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

snr = (df.flux.abs() / df.flux_err).replace([np.inf, -np.inf], np.nan).dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(snr, 0, 30), bins=120, color='#C44E52')
ax.axvline(config.SNR_MIN, color='k', lw=1.5, ls='--', label=f'SNR_MIN = {config.SNR_MIN}')
ax.set_yscale('log')
ax.set_xlabel('|flux| / error (clipped at 30)'); ax.set_ylabel('Number of observations (log)')
ax.set_title(f'SNR per observation · {(snr < config.SNR_MIN).mean():.0%} below threshold')
ax.legend(); plt.tight_layout()
fig.savefig(FIGDIR / '01_snr.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Temporal and passband coverage


In [ ]:

det = df.groupby('object_id').size()
nb = df.groupby('object_id')['passband'].nunique()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(np.clip(det, 0, 400), bins=50, color='#4C72B0')
ax[0].axvline(config.MIN_OBS, color='k', ls='--', lw=1.2, label=f'MIN_OBS = {config.MIN_OBS}')
ax[0].set_xlabel('Observations per object'); ax[0].set_ylabel('Number of objects')
ax[0].set_title('Light-curve length'); ax[0].legend()
ax[1].bar(*zip(*sorted(nb.value_counts().items())), color='#8172B3')
ax[1].set_xlabel('Number of observed bands'); ax[1].set_ylabel('Number of objects')
ax[1].set_title('Spectral coverage per object')
plt.tight_layout()
fig.savefig(FIGDIR / '01_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Training–test distribution shift


In [ ]:

test_tt = pd.read_csv(config.DIR_RAW / 'plasticc_test_metadata.csv.gz',
                      usecols=['true_target']).true_target
tr = metadata_table.target.value_counts(normalize=True)
te = test_tt.value_counts(normalize=True)
compartidas = sorted(set(tr.index))                      
labels = [CLASS_NAMES.get(c, c) for c in compartidas]
x = np.arange(len(compartidas)); w = 0.4
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - w/2, [tr.get(c, 0) for c in compartidas], w, label='train', color='#4C72B0')
ax.bar(x + w/2, [te.get(c, 0) for c in compartidas], w, label='test',  color='#DD8452')
frac_novel = te[[c for c in te.index if c not in compartidas]].sum()
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Fraction of the dataset'); ax.set_yscale('log')
ax.set_title(f'Class composition: training vs test  '
             f'(+{frac_novel:.1%} of test objects belong to novel 99x classes absent from training)')
ax.legend(); plt.tight_layout()
fig.savefig(FIGDIR / '01_train_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Interpretation

Training and test data differ in class prevalence, depth and temporal coverage.
Internal cross-validation therefore does not fully characterize external performance.
The following notebooks keep preprocessing and representation definitions fixed.
